# Zero-Shot PageRank Prediction Across Graphs via Task-Tree GNNs
## + Hierarchical cluster-and-merge extension (proposal implementation)

Inspired by *Towards Graph Foundation Models: Learning Generalities Across Graphs via Task-Trees* (Wang et al., 2025).

**Part 0 (original notebook, unchanged logic):** a GraphSAGE task-tree encoder trained on 32 Erdős–Rényi graphs.

**Parts 1–7 (new, following the proposal's "Suggested order of work"):**

| Step | What this notebook adds |
|---|---|
| 1 | In-degree and power-iteration baselines, a much larger unseen test suite (5 graph families, several sizes, optional SNAP graph), top-k / NDCG metrics |
| 2 | Size-independent features, log-scale target `log(N·PR)`, ranking loss |
| 3 | `PageRankPropGNN`: sum aggregation weighted by 1/out-degree (the PageRank operator), 25 weight-shared propagation steps |
| 4 | Training on mixed graph families (ER, SBM, preferential attachment, configuration model, scale-free); held-out evaluation |
| 5 | Cluster → merge → repeat hierarchy (Louvain + recursive splitting), exact-aggregation sanity check, local-PageRank oracle |
| 6 | The same GNN plugged into every level; training Stages A (warm-up), B (own mistakes), C (end-to-end); final PageRank refinement |
| 7 | Per-level error, ablations (stages, refinement, cluster size), speed (iterations to target accuracy, wall clock) |

**How to run:** Kernel → Restart & Run All. Real SNAP graphs are downloaded automatically into `data/` (Cell 9),
every metric is saved under `results/`, and the last cell compares the **base code** (original GraphSAGE) with the
**new additions** (flat PRProp-GNN, hierarchical GNN) on real and synthetic graphs.
Set `CFG['quick'] = True` in Cell 2 for a fast smoke test (about 5–10 minutes) before the full run.

In [ ]:
# Cell 1: Dependency Installation
# PyTorch itself is NOT reinstalled here (that would replace a CUDA build with the CPU one).
# If torch is missing, install the build for your CUDA version from https://pytorch.org first.
%pip install -q torch_geometric networkx scipy numpy matplotlib pandas

In [ ]:
# Cell 2: Imports, Seed Setting & Global Config
import random, time, copy, os, math, gzip, shutil, urllib.request, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
from scipy.stats import ConstantInputWarning
warnings.filterwarnings('ignore', category=ConstantInputWarning)
import numpy as np
import networkx as nx
import scipy.sparse as sp
from scipy.sparse.csgraph import reverse_cuthill_mckee
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv
from scipy.stats import spearmanr, kendalltau
import matplotlib.pyplot as plt

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

CFG = dict(
    quick=False,                 # True -> tiny sizes / few epochs, for a smoke test
    alpha=0.85,                  # PageRank damping
    base_epochs=500,             # original GraphSAGE training epochs
    # --- Step 1/4: evaluation suite ---
    families=['er', 'sbm', 'pa', 'config', 'scalefree'],
    test_sizes=(5000, 30000),    # held-out graphs per family at these sizes
    data_dir='data', results_dir='results',
    # Real SNAP graphs: downloaded automatically (or put the file in data/ yourself). Set enabled=False to skip one.
    snap_datasets=[
        dict(name='AS-733 (as20000102)', file='as20graph.txt', enabled=True,
             url='https://raw.githubusercontent.com/snap-stanford/snap/master/examples/as20graph.txt'),
        dict(name='C. elegans frontal', file='C-elegans-frontal.txt', enabled=True,
             url='https://raw.githubusercontent.com/snap-stanford/snap/master/examples/localmotifcluster/C-elegans-frontal.txt'),
        dict(name='wiki-Vote', file='wiki-Vote.txt', enabled=True,
             url='https://snap.stanford.edu/data/wiki-Vote.txt.gz'),
        dict(name='p2p-Gnutella31', file='p2p-Gnutella31.txt', enabled=True,
             url='https://snap.stanford.edu/data/p2p-Gnutella31.txt.gz'),
        dict(name='cit-HepPh', file='cit-HepPh.txt', enabled=True,
             url='https://snap.stanford.edu/data/cit-HepPh.txt.gz'),
        dict(name='web-Google', file='web-Google.txt', enabled=False,      # 875k nodes: slow; enable for the scale test
             url='https://snap.stanford.edu/data/web-Google.txt.gz'),
    ],
    # --- Step 2-4: flat PageRankPropGNN ---
    hidden=64, K_prop=25, dropout=0.1,
    flat_train_per_family=8, flat_train_n=(300, 2000), flat_epochs=150,
    rank_weight=0.5, rank_pairs=4096,
    # --- Step 5-7: hierarchy ---
    max_cluster=50, cluster_method='auto',      # 'louvain', 'rcm' (fast, no community detection) or 'auto'
    louvain_max_nodes=100_000,                   # 'auto' uses Louvain up to this size, RCM above it
    ablation_max_nodes=40_000,                   # cluster-size ablation only on graphs up to this size
    hier_train_per_family=3, hier_train_n=(2000, 6000),
    stageA_epochs=40, stageB_rounds=3, stageB_epochs=15, stageC_epochs=25,
    level_loss_weight=0.3, refine_iters=(0, 10, 20), target_err=1e-6,
    ablation_cluster_sizes=(20, 50, 100),
)
if CFG['quick']:
    CFG.update(base_epochs=30, test_sizes=(1500, 4000), flat_train_per_family=3, flat_train_n=(200, 600),
               flat_epochs=10, hier_train_per_family=1, hier_train_n=(800, 1500),
               stageA_epochs=4, stageB_rounds=1, stageB_epochs=2, stageC_epochs=2)
    for d in CFG['snap_datasets']:                 # quick mode: only the two small real graphs
        d['enabled'] = d['enabled'] and d['file'] in ('as20graph.txt', 'C-elegans-frontal.txt')
ALPHA = CFG['alpha']
os.makedirs(CFG['data_dir'], exist_ok=True); os.makedirs(CFG['results_dir'], exist_ok=True)

# Plot colours (fixed categorical order, one colour per method everywhere)
COLORS = {'In-degree': '#9c9c9c', 'Power iter (1 step)': '#f2a600', 'Flat GraphSAGE (base)': '#f47230',
          'Flat PRProp-GNN': '#257de0', 'Hier: local-PR oracle': '#ee7fac', 'Hier-GNN': '#00b894',
          f"Power iter ({CFG['K_prop']} steps)": '#008a17'}

In [ ]:
# Cell 3: Structural Feature Extraction & Graph Dataset Builder (original)
def extract_structural_features(G):
    """
    Projects non-attributed graph nodes into a shared structural feature space
    (In-degree, Out-degree, Normalized Degrees, Local Clustering Coefficient).
    """
    num_nodes = G.number_of_nodes()
    in_degrees = dict(G.in_degree())
    out_degrees = dict(G.out_degree())
    clustering = nx.clustering(G.to_undirected())

    features = []
    for i in range(num_nodes):
        in_d = in_degrees[i]
        out_d = out_degrees[i]
        norm_in_d = in_d / max(num_nodes - 1, 1)
        norm_out_d = out_d / max(num_nodes - 1, 1)
        c = clustering[i]
        features.append([in_d, out_d, norm_in_d, norm_out_d, c])
    return torch.tensor(features, dtype=torch.float)

def create_er_graph_data(num_nodes, p, alpha=0.85):
    """
    Generates a single directed Erdős–Rényi graph G(N, p),
    calculates exact PageRank ground truth, and prepares PyG Data object.
    Target y = N * PageRank (scale-invariant relative importance).
    """
    G = nx.erdos_renyi_graph(num_nodes, p, directed=True, seed=random.randint(0, 100000))
    pr_dict = nx.pagerank(G, alpha=alpha, max_iter=200)
    pr_values = np.array([pr_dict[i] for i in range(num_nodes)], dtype=np.float32)
    scaled_pr = pr_values * num_nodes

    x = extract_structural_features(G)
    edges = list(G.edges())
    if len(edges) > 0:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)

    y = torch.tensor(scaled_pr, dtype=torch.float).unsqueeze(1)
    data = Data(x=x, edge_index=edge_index, y=y, num_nodes=num_nodes)
    return data, G

In [ ]:
# Cell 4: Generate 32 Seen Training Graphs (original)
# (The unseen test graphs now live in the shared test suite, Cell 9.)
print("Generating 32 Seen Training Erdős–Rényi Graphs...")
train_graphs = []
for i in range(32):
    n_nodes = random.randint(500, 1000)
    prob = random.uniform(0.04, 0.12)
    data, _ = create_er_graph_data(n_nodes, prob)
    train_graphs.append(data)

train_loader = DataLoader(train_graphs, batch_size=4, shuffle=True)
print(f"Successfully built {len(train_graphs)} training graphs.")

In [ ]:
# Cell 5: GraphSAGE Architecture with Multi-Scale Jumping Knowledge (original)
class TaskTreeGraphSAGE(nn.Module):
    def __init__(self, in_channels=5, hidden_channels=128, out_channels=1, num_layers=4, dropout=0.15):
        super().__init__()
        self.num_layers = num_layers
        self.dropout = dropout
        self.input_proj = nn.Sequential(
            nn.Linear(in_channels, hidden_channels),
            nn.LayerNorm(hidden_channels),
            nn.ReLU()
        )
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(SAGEConv(hidden_channels, hidden_channels, aggr='mean'))
            self.norms.append(nn.LayerNorm(hidden_channels))
        combined_dim = hidden_channels * (num_layers + 1)
        self.head = nn.Sequential(
            nn.Linear(combined_dim, hidden_channels),
            nn.LayerNorm(hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Linear(hidden_channels // 2, out_channels),
            nn.Softplus()
        )

    def forward(self, x, edge_index):
        h0 = self.input_proj(x)
        layer_outputs = [h0]
        h = h0
        for i in range(self.num_layers):
            h_conv = self.convs[i](h, edge_index)
            h_conv = self.norms[i](h_conv)
            h_conv = F.relu(h_conv)
            h = h + h_conv
            h = F.dropout(h, p=self.dropout, training=self.training)
            layer_outputs.append(h)
        h_multi_scale = torch.cat(layer_outputs, dim=-1)
        return self.head(h_multi_scale)

model = TaskTreeGraphSAGE(in_channels=5, hidden_channels=128, out_channels=1, num_layers=4, dropout=0.15).to(device)
print(model)

In [ ]:
# Cell 6: Model Training on 32 Seen Graphs (original)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=1e-5)
criterion = nn.SmoothL1Loss()
epochs = CFG['base_epochs']

loss_history = []
model.train()
print("Beginning training on 32 seen Erdős–Rényi graphs...")
for epoch in range(1, epochs + 1):
    total_loss = 0.0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    avg_loss = total_loss / len(train_graphs)
    loss_history.append(avg_loss)
    if epoch % 15 == 0 or epoch == 1:
        print(f"Epoch [{epoch:03d}/{epochs:03d}] | Smooth L1 Loss: {avg_loss:.5f}")
base_model = model

---
# Step 1 — Baselines and a real test suite
Shared utilities: fast sparse PageRank (with warm starts), five directed graph families, metrics
(Spearman, Kendall, top-10 %, precision@100, NDCG@100, L1), and baselines.

In [ ]:
# Cell 7: Graph families + sparse PageRank
def edges_to_csr(n, src, dst):
    """Binary directed adjacency (u -> v), no self-loops, no multi-edges."""
    src = np.asarray(src, dtype=np.int64); dst = np.asarray(dst, dtype=np.int64)
    keep = src != dst
    A = sp.csr_matrix((np.ones(int(keep.sum())), (src[keep], dst[keep])), shape=(n, n))
    A.sum_duplicates()
    A.data[:] = 1.0
    return A

def gen_er(n, avg_deg, rng):
    m = rng.binomial(n * (n - 1), min(avg_deg / (n - 1), 1.0))
    return edges_to_csr(n, rng.integers(0, n, m), rng.integers(0, n, m))

def gen_sbm(n, avg_deg, rng, mix=0.1):
    """Directed stochastic block model with 4-20 communities of random sizes."""
    k = int(rng.integers(4, 21))
    sizes = np.maximum((rng.dirichlet(np.full(k, 2.0)) * n).astype(int), 5)
    sizes[-1] = max(n - sizes[:-1].sum(), 5)
    n = int(sizes.sum()); off = np.r_[0, np.cumsum(sizes)]
    src, dst = [], []
    for a in range(k):
        for b in range(k):
            p = avg_deg * (1 - mix) / sizes[a] if a == b else avg_deg * mix / max(n - sizes[a], 1)
            m = rng.binomial(int(sizes[a]) * int(sizes[b]), min(p, 1.0))
            src.append(off[a] + rng.integers(0, sizes[a], m)); dst.append(off[b] + rng.integers(0, sizes[b], m))
    perm = rng.permutation(n)            # hide the block order
    return edges_to_csr(n, perm[np.concatenate(src)], perm[np.concatenate(dst)])

def gen_pa(n, m_out, rng, p_uniform=0.2, p_back=0.1):
    """Citation-like preferential attachment: new node cites ~m_out older nodes, chosen ∝ in-degree+1."""
    m0 = m_out + 1
    cap = int(4 * m_out)
    pool = np.empty(m0 + n * (cap + 1), dtype=np.int64); pool[:m0] = np.arange(m0); plen = m0
    src, dst = [], []
    for t in range(m0, n):
        k = int(min(1 + rng.poisson(m_out - 1), cap))
        tg = np.where(rng.random(k) < p_uniform, rng.integers(0, t, k), pool[rng.integers(0, plen, k)])
        tg = np.unique(tg)
        src.append(np.full(len(tg), t)); dst.append(tg)
        pool[plen:plen + len(tg)] = tg; plen += len(tg)
        pool[plen] = t; plen += 1
    src = np.concatenate(src); dst = np.concatenate(dst)
    back = rng.random(len(src)) < p_back       # some reciprocal links so it is not a DAG
    return edges_to_csr(n, np.r_[src, dst[back]], np.r_[dst, src[back]])

def gen_config(n, avg_deg, rng, gamma_in=2.2, gamma_out=2.6):
    """Directed configuration model with power-law in/out degree sequences."""
    def seq(gamma):
        w = rng.pareto(gamma - 1, n) + 1
        w = np.minimum(w, n / 10)
        return np.minimum(rng.poisson(w * avg_deg / w.mean()), n - 1)
    din, dout = seq(gamma_in), seq(gamma_out)
    m = int(min(din.sum(), dout.sum()))
    src = rng.permutation(np.repeat(np.arange(n), dout))[:m]
    dst = rng.permutation(np.repeat(np.arange(n), din))[:m]
    return edges_to_csr(n, src, dst)

def gen_scalefree(n, rng):
    G = nx.scale_free_graph(n, alpha=0.41, beta=0.54, gamma=0.05,
                            delta_in=float(rng.uniform(0.1, 1.0)), delta_out=0.0, seed=int(rng.integers(1 << 30)))
    e = np.array(list(G.edges()), dtype=np.int64).reshape(-1, 2)
    return edges_to_csr(n, e[:, 0], e[:, 1])

def make_graph(family, n, rng):
    if family == 'er':        return gen_er(n, rng.uniform(3, 30), rng)
    if family == 'sbm':       return gen_sbm(n, rng.uniform(4, 20), rng, mix=rng.uniform(0.05, 0.3))
    if family == 'pa':        return gen_pa(n, int(rng.integers(2, 9)), rng)
    if family == 'config':    return gen_config(n, rng.uniform(3, 15), rng, rng.uniform(2.1, 3.0), rng.uniform(2.1, 3.0))
    if family == 'scalefree': return gen_scalefree(n, rng)
    raise ValueError(family)

def fetch_snap(ds, data_dir=None):
    """Return the local path of a SNAP edge list, downloading (and un-gzipping) it if needed."""
    data_dir = data_dir or CFG['data_dir']
    path = os.path.join(data_dir, ds['file'])
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    print(f"  downloading {ds['name']} from {ds['url']} ...")
    tmp = path + ('.gz' if ds['url'].endswith('.gz') else '.part')
    try:
        req = urllib.request.Request(ds['url'], headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=120) as r, open(tmp, 'wb') as f:
            shutil.copyfileobj(r, f)
        if tmp.endswith('.gz'):
            with gzip.open(tmp, 'rb') as fi, open(path, 'wb') as fo:
                shutil.copyfileobj(fi, fo)
            os.remove(tmp)
        else:
            os.replace(tmp, path)
    except Exception as e:
        if os.path.exists(tmp): os.remove(tmp)
        raise RuntimeError(f"Could not download {ds['name']} ({e}). Download {ds['url']} manually, "
                           f"un-gzip it if needed, and save it as {path} — or set enabled=False for it in CFG.") from e
    return path

def load_snap(path):
    e = np.loadtxt(path, comments='#', dtype=np.int64).reshape(-1, 2)
    ids, inv = np.unique(e, return_inverse=True)
    inv = inv.reshape(-1, 2)
    return edges_to_csr(len(ids), inv[:, 0], inv[:, 1])

def clustering_undirected(A, chunk=2000):
    """Local clustering coefficient of the undirected version of A (same as nx.clustering(G.to_undirected()))."""
    U = ((A + A.T) > 0).astype(np.float64).tocsr(); U.setdiag(0); U.eliminate_zeros()
    d = np.asarray(U.sum(1)).ravel()
    tri2 = np.zeros(U.shape[0])
    for s in range(0, U.shape[0], chunk):          # row chunks keep U@U small on hub-heavy graphs
        R = U[s:s + chunk]
        tri2[s:s + chunk] = np.asarray((R @ U).multiply(R).sum(1)).ravel()
    return np.divide(tri2, d * (d - 1), out=np.zeros_like(tri2), where=d > 1)

def transition(A):
    """Row-stochastic P = D_out^-1 A (dangling rows stay zero; their mass leaks to v)."""
    out = np.asarray(A.sum(1)).ravel()
    inv = np.divide(1.0, out, out=np.zeros_like(out, dtype=float), where=out > 0)
    return (sp.diags(inv) @ A).tocsr()

def pagerank_power(P, v, alpha=ALPHA, x0=None, tol=1e-10, max_iter=2000, ref=None, PT=None):
    """
    x <- alpha * x P + (alpha * leak·x + 1 - alpha) v, where leak = 1 - rowsum(P)
    (dangling / leaked mass is redistributed by v, exactly like nx.pagerank).
    Works for any sub-stochastic P, so it also runs on the weighted coarse graphs.
    Returns (x, iterations, history of L1 distance to `ref` if given).
    """
    PT = P.T.tocsr() if PT is None else PT
    leak = np.clip(1.0 - np.asarray(P.sum(1)).ravel(), 0.0, 1.0)
    x = (v.copy() if x0 is None else np.asarray(x0, dtype=float) / np.sum(x0))
    hist = [] if ref is None else [np.abs(x - ref).sum()]
    it = 0
    for it in range(1, max_iter + 1):
        x_new = alpha * (PT @ x) + (alpha * leak.dot(x) + 1.0 - alpha) * v
        x_new /= x_new.sum()
        err = np.abs(x_new - x).sum()
        x = x_new
        if ref is not None: hist.append(np.abs(x - ref).sum())
        if err < tol: break
    return x, it, hist

def make_record(name, family, A):
    t = time.time()
    n = A.shape[0]
    P = transition(A); v = np.full(n, 1.0 / n)
    pr, _, _ = pagerank_power(P, v, tol=1e-13)
    return dict(name=name, family=family, n=n, m=A.nnz, A=A, P=P, v=v, pr=pr, pr_time=time.time() - t)

In [ ]:
# Cell 8: Metrics, baselines and the results table
def ndcg_at_k(pred, true, k):
    k = min(k, len(true))
    top = np.argsort(-pred, kind='stable')[:k]
    disc = 1.0 / np.log2(np.arange(2, k + 2))
    return float((true[top] * disc).sum() / (np.sort(true)[::-1][:k] * disc).sum())

def topk_overlap(pred, true, k):
    k = min(k, len(true))
    return len(set(np.argpartition(-true, k - 1)[:k]) & set(np.argpartition(-pred, k - 1)[:k])) / k

def rank_metrics(pred, true):
    pred = np.asarray(pred, dtype=float)
    p = pred / pred.sum() if pred.min() >= 0 and pred.sum() > 0 else pred
    n = len(true)
    return dict(spearman=spearmanr(pred, true)[0], kendall=kendalltau(pred, true)[0],
                top10pct=topk_overlap(pred, true, max(int(0.10 * n), 3)),
                p_at_100=topk_overlap(pred, true, 100), ndcg_at_100=ndcg_at_k(pred, true, 100),
                l1=float(np.abs(p - true).sum()), mse=float(np.mean((p - true) ** 2)))

def baseline_indegree(rec):
    return np.asarray(rec.A.sum(0)).ravel() + 1e-12

def baseline_power(rec, k):
    x, _, _ = pagerank_power(rec['P'], rec['v'], max_iter=k, tol=0.0)
    return x

def graph_kind(rec):
    return 'real' if rec['family'] == 'real' else ('synthetic: original ER' if rec['family'] == 'er-orig'
                                                     else 'synthetic: held-out families')

RESULTS = []
def log_result(rec, method, pred, **extra):
    row = dict(graph=rec['name'], family=rec['family'], kind=graph_kind(rec), n=rec['n'], m=rec['m'], method=method,
               **rank_metrics(pred, rec['pr']), **extra)
    RESULTS.append(row)
    return row

def results_table(metric='spearman', methods=None, extra_filter=None):
    df = pd.DataFrame(RESULTS)
    if extra_filter is not None: df = df[extra_filter(df)]
    if methods is not None: df = df[df.method.isin(methods)]
    return df.pivot_table(index=['family', 'n', 'graph'], columns='method', values=metric, aggfunc='last').round(4)

# Small dict-like access so both rec['A'] and rec.A work
class Rec(dict):
    __getattr__ = dict.__getitem__

In [ ]:
# Cell 9: The unseen test suite (Step 1: "more test graphs")
rng_test = np.random.default_rng(2026)
test_suite = []
# (a) the original ER configs (including the commented-out ones) + the N=400 graph
for n, p in [(400, 0.02), (500, 0.02), (600, 0.05), (700, 0.04), (800, 0.08), (900, 0.03)]:
    test_suite.append(Rec(make_record(f"ER_N{n}_p{p}", 'er-orig', gen_er(n, p * (n - 1), rng_test))))
# (b) held-out graphs of every family, at sizes far beyond the training range
for fam in CFG['families']:
    for n in CFG['test_sizes']:
        test_suite.append(Rec(make_record(f"{fam}_N{n}", fam, make_graph(fam, n, rng_test))))
# (c) real SNAP graphs — downloaded automatically into CFG['data_dir']; a failed download stops the run with
#     instructions instead of silently skipping, so the comparison always includes real graphs.
for ds in CFG['snap_datasets']:
    if not ds['enabled']:
        continue
    t = time.time()
    test_suite.append(Rec(make_record(ds['name'], 'real', load_snap(fetch_snap(ds)))))
    print(f"  loaded {ds['name']}: {test_suite[-1].n:,} nodes, {test_suite[-1].m:,} edges ({time.time() - t:.1f}s)")
assert any(r.family == 'real' for r in test_suite), "No real SNAP graph loaded — enable one in CFG['snap_datasets']"

print(pd.DataFrame([dict(graph=r.name, kind=graph_kind(r), n=r.n, edges=r.m, avg_deg=round(r.m / r.n, 2),
                         max_in=int(r.A.sum(0).max()), dangling=int((r.A.sum(1) == 0).sum()))
                    for r in test_suite]).to_string(index=False))

In [ ]:
# Cell 10: Step 1 — evaluate the original model against simple baselines
def legacy_features(A):
    n = A.shape[0]
    ind = np.asarray(A.sum(0)).ravel(); outd = np.asarray(A.sum(1)).ravel()
    X = np.column_stack([ind, outd, ind / max(n - 1, 1), outd / max(n - 1, 1), clustering_undirected(A)])
    return torch.tensor(X, dtype=torch.float)

@torch.no_grad()
def predict_base(model, rec):
    model.eval()
    ei = torch.tensor(np.vstack(rec.A.nonzero()), dtype=torch.long, device=device)
    out = model(legacy_features(rec.A).to(device), ei).squeeze(-1).cpu().numpy().astype(float)
    return out / (out.sum() + 1e-12)

K_PROP = CFG['K_prop']
for rec in test_suite:
    log_result(rec, 'In-degree', baseline_indegree(rec))
    log_result(rec, 'Power iter (1 step)', baseline_power(rec, 1))
    log_result(rec, f'Power iter ({K_PROP} steps)', baseline_power(rec, K_PROP))
    log_result(rec, 'Flat GraphSAGE (base)', predict_base(base_model, rec))

print("Spearman rho")
print(results_table('spearman').to_string())
print("\nPrecision@100")
print(results_table('p_at_100').to_string())

In [ ]:
# Cell 11: Visualizing Convergence and Predictions (original Cell 8, fixed)
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
axs[0].plot(range(1, len(loss_history) + 1), loss_history, color='#257de0', lw=2)
axs[0].set_title("Training loss (Smooth L1) across 32 ER graphs")
axs[0].set_xlabel("Epoch"); axs[0].set_ylabel("Loss"); axs[0].grid(True, alpha=0.3)

# The old cell crashed because it looked for an N=1000 graph that no longer existed.
# Pick the largest original-ER test graph instead of hard-coding N.
sample_rec = max([r for r in test_suite if r.family == 'er-orig'], key=lambda r: r.n)
sample_pred = predict_base(base_model, sample_rec)
rho = spearmanr(sample_pred, sample_rec.pr)[0]
axs[1].scatter(sample_rec.pr, sample_pred, alpha=0.6, color='#257de0', edgecolors='white', linewidths=0.5, s=35)
lo, hi = min(sample_rec.pr.min(), sample_pred.min()), max(sample_rec.pr.max(), sample_pred.max())
axs[1].plot([lo, hi], [lo, hi], color='#666666', ls='--', lw=1.5, label='Ideal 1:1')
axs[1].set_title(f"Unseen graph ({sample_rec.name}) | rho = {rho:.3f}")
axs[1].set_xlabel("Ground-truth PageRank"); axs[1].set_ylabel("Zero-shot predicted PageRank")
axs[1].legend(); axs[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
# Steps 2–4 — Size-independent features, the PageRank operator, mixed graph families

**Features (12, all size-independent).** For a node *u* on a (possibly coarse, weighted) graph with transition
matrix *P*, teleport vector *v* and a cluster assignment:
`log(1+in-deg)`, `log(1+out-deg)`, `log(1+in-deg / mean in-deg)`, in-degree percentile within the cluster,
clustering coefficient (intra-cluster), `log(1+Σ_w N·v_w·P_wu)` (= Σ 1/out-deg over in-neighbours on the
original graph — the one-hop PageRank hint), the same split into external and intra-cluster inflow,
`log(N·v_u)` (relative size of a super node), self-retention `P_uu`, leak `1-Σ_v P_uv` (dangling), `log(cluster size)`.

**Model.** `PageRankPropGNN`: encoder → 25 iterations of `m_v = Σ_u P_uv · W h_u` (sum aggregation weighted by
1/out-degree, i.e. the PageRank operator) with **one shared weight set** → head. The output is a score whose
softmax inside a group (a whole graph, or a cluster) is that node's share; the loss is Smooth-L1 on
`log(N·PR)` plus a pairwise ranking loss.

In [ ]:
# Cell 12: Level feature builder (used by the flat model AND every hierarchy level)
N_FEATS = 12
FEAT_NAMES = ['log_in', 'log_out', 'log_in_rel', 'in_pct', 'clust', 'log_inflow', 'log_ext_in',
              'log_intra_in', 'log_rel_tel', 'self_ret', 'leak', 'log_csize']

def level_features(P, v, labels, flat=False):
    """
    P: (n,n) sub-stochastic CSR transition matrix of this level; v: teleport vector; labels: cluster of each node.
    Returns (features [n, 12] float32, message edges (src, dst, weight) restricted to intra-cluster, off-diagonal).
    """
    n = P.shape[0]
    coo = P.tocoo()
    off = coo.row != coo.col
    r, c, w = coo.row[off], coo.col[off], coo.data[off]
    intra = labels[r] == labels[c]
    nv = n * v
    in_d = np.bincount(c, minlength=n).astype(float)
    out_d = np.bincount(r, minlength=n).astype(float)
    flow = w * nv[r]
    inflow = np.bincount(c, weights=flow, minlength=n)
    intra_in = np.bincount(c[intra], weights=flow[intra], minlength=n)
    ext_in = np.maximum(inflow - intra_in, 0.0)
    leak = np.clip(1.0 - np.asarray(P.sum(1)).ravel(), 0.0, 1.0)
    counts = np.bincount(labels)
    gsize = counts[labels].astype(float)
    order = np.lexsort((in_d, labels))                       # within-cluster in-degree percentile
    pos = np.empty(n, dtype=np.int64); pos[order] = np.arange(n)
    cstart = np.r_[0, np.cumsum(counts)][:-1][labels]
    pct = (pos - cstart) / np.maximum(gsize - 1, 1)
    Ui = sp.csr_matrix((np.ones(int(intra.sum())), (r[intra], c[intra])), shape=(n, n))
    clust = clustering_undirected(Ui)
    X = np.column_stack([np.log1p(in_d), np.log1p(out_d), np.log1p(in_d / max(in_d.mean(), 1e-9)), pct, clust,
                         np.log1p(inflow), np.log1p(ext_in), np.log1p(intra_in), np.log(np.maximum(nv, 1e-12)),
                         P.diagonal(), leak, np.log(gsize)]).astype(np.float32)
    if flat:
        X[:, [6, 11]] = 0.0                                  # no cluster context in flat mode
    return X, (r[intra], c[intra], w[intra])

def make_batch(P, v, labels, target_logshare=None, flat=False):
    """Numpy batch for one level (or one flat graph). Groups = clusters."""
    X, (r, c, w) = level_features(P, v, labels, flat=flat)
    labels = np.asarray(labels, dtype=np.int64)
    G = int(labels.max()) + 1
    return dict(x=X, src=r.astype(np.int64), dst=c.astype(np.int64), ew=w.astype(np.float32), g=labels, G=G,
                tgt=None if target_logshare is None else np.asarray(target_logshare, dtype=np.float32))

def collate(batches):
    """Disjoint union of several level batches -> one torch batch on `device`."""
    xs, src, dst, ew, g, tgt = [], [], [], [], [], []
    n_off = g_off = 0
    for b in batches:
        xs.append(b['x']); src.append(b['src'] + n_off); dst.append(b['dst'] + n_off); ew.append(b['ew'])
        g.append(b['g'] + g_off); tgt.append(b['tgt'] if b['tgt'] is not None else np.zeros(len(b['x']), np.float32))
        n_off += len(b['x']); g_off += b['G']
    g = np.concatenate(g)
    counts = np.bincount(g, minlength=g_off)
    T = lambda a, dt: torch.as_tensor(a, dtype=dt, device=device)
    return dict(x=T(np.concatenate(xs), torch.float), ei=T(np.vstack([np.concatenate(src), np.concatenate(dst)]), torch.long),
                ew=T(np.concatenate(ew), torch.float), g=T(g, torch.long), G=g_off,
                gsize=T(counts[g], torch.long), gstart=T(np.r_[0, np.cumsum(counts)][:-1], torch.long),
                gorder=T(np.argsort(g, kind='stable'), torch.long), tgt=T(np.concatenate(tgt), torch.float))

In [ ]:
# Cell 13: PageRankPropGNN, group softmax and losses
class PageRankPropGNN(nn.Module):
    """
    Encoder -> K weight-shared propagation steps along the PageRank operator -> score.
    m_v = Σ_u P_uv · W_msg h_u  (sum aggregation weighted by transition probability = 1/out-degree on the raw graph)
    """
    def __init__(self, in_dim=N_FEATS, hidden=64, K=25, dropout=0.1):
        super().__init__()
        self.K, self.dropout = K, dropout
        self.enc = nn.Sequential(nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.msg = nn.Linear(hidden, hidden, bias=False)
        self.upd = nn.Linear(3 * hidden, hidden)
        self.norm = nn.LayerNorm(hidden)
        self.head = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, x, ei, ew, K=None):
        h0 = self.enc(x)
        h = h0
        src, dst = ei[0], ei[1]
        for _ in range(K or self.K):
            m = torch.zeros_like(h).index_add_(0, dst, self.msg(h)[src] * ew.unsqueeze(1))
            m = torch.sign(m) * torch.log1p(m.abs())           # keep hub inflows on a sane scale
            h = self.norm(h + F.relu(self.upd(torch.cat([h, m, h0], dim=-1))))
            h = F.dropout(h, p=self.dropout, training=self.training)
        return self.head(torch.cat([h, h0], dim=-1)).squeeze(-1)

def group_logsoftmax(o, g, G):
    mx = torch.full((G,), float('-inf'), device=o.device).scatter_reduce(0, g, o.detach(), 'amax', include_self=True)
    z = torch.exp(o - mx[g])
    s = torch.zeros(G, device=o.device, dtype=o.dtype).index_add(0, g, z)
    return o - mx[g] - torch.log(s[g])

def ranking_loss(score, tgt, b, n_pairs):
    """Logistic pairwise loss on random node pairs drawn from the same group."""
    valid = torch.nonzero(b['gsize'] > 1).squeeze(-1)
    if valid.numel() == 0:
        return score.sum() * 0.0
    i = valid[torch.randint(len(valid), (n_pairs,), device=score.device)]
    gi = b['g'][i]
    j = b['gorder'][b['gstart'][gi] + (torch.rand(n_pairs, device=score.device) * b['gsize'][i]).long()]
    d = tgt[i] - tgt[j]
    mask = d.abs() > 0.05
    if mask.sum() == 0:
        return score.sum() * 0.0
    return F.softplus(-torch.sign(d[mask]) * (score[i][mask] - score[j][mask])).mean()

def share_loss(lp, b, rank_weight=CFG['rank_weight'], n_pairs=CFG['rank_pairs']):
    """lp = predicted log-share inside each group. Smooth-L1 on log(N·PR) (constant shift cancels) + ranking."""
    mask = b['gsize'] > 1
    if mask.sum() == 0:
        return lp.sum() * 0.0
    return F.smooth_l1_loss(lp[mask], b['tgt'][mask]) + rank_weight * ranking_loss(lp, b['tgt'], b, n_pairs)

def train_on_batches(model, batches, epochs, opt, node_budget=40000, tag=''):
    hist = []
    for ep in range(1, epochs + 1):
        model.train()
        idx = np.random.permutation(len(batches)); tot = 0.0; nb = 0
        chunk, size = [], 0
        for k, i in enumerate(idx):
            chunk.append(batches[i]); size += len(batches[i]['x'])
            if size >= node_budget or k == len(idx) - 1:
                b = collate(chunk)
                lp = group_logsoftmax(model(b['x'], b['ei'], b['ew']), b['g'], b['G'])
                loss = share_loss(lp, b)
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
                tot += loss.item(); nb += 1; chunk, size = [], 0
        hist.append(tot / max(nb, 1))
        if ep == 1 or ep % max(1, epochs // 10) == 0 or ep == epochs:
            print(f"{tag} epoch {ep:03d}/{epochs} | loss {hist[-1]:.4f}")
    return hist

@torch.no_grad()
def predict_flat(model, rec):
    model.eval()
    labels = np.zeros(rec.n, dtype=np.int64)
    b = collate([make_batch(rec.P, rec.v, labels, flat=True)])
    lp = group_logsoftmax(model(b['x'], b['ei'], b['ew']), b['g'], b['G'])
    return torch.exp(lp).double().cpu().numpy()

In [ ]:
# Cell 14: Step 4 — train the flat PageRankPropGNN on mixed graph families
rng_train = np.random.default_rng(7)
flat_train = []
for fam in CFG['families']:
    for _ in range(CFG['flat_train_per_family']):
        n = int(rng_train.integers(*CFG['flat_train_n']))
        flat_train.append(Rec(make_record(f"train_{fam}", fam, make_graph(fam, n, rng_train))))
flat_batches = [make_batch(r.P, r.v, np.zeros(r.n, np.int64), np.log(r.pr), flat=True) for r in flat_train]
print(f"{len(flat_train)} flat training graphs, {sum(r.n for r in flat_train)} nodes total")

set_seed(1)
flat_model = PageRankPropGNN(hidden=CFG['hidden'], K=K_PROP, dropout=CFG['dropout']).to(device)
opt_flat = torch.optim.AdamW(flat_model.parameters(), lr=2e-3, weight_decay=1e-5)
flat_hist = train_on_batches(flat_model, flat_batches, CFG['flat_epochs'], opt_flat, node_budget=4000, tag='[flat]')

In [ ]:
# Cell 15: Evaluate the flat PRProp-GNN on the unseen suite
for rec in test_suite:
    log_result(rec, 'Flat PRProp-GNN', predict_flat(flat_model, rec))

flat_methods = ['In-degree', 'Power iter (1 step)', 'Flat GraphSAGE (base)', 'Flat PRProp-GNN', f'Power iter ({K_PROP} steps)']
for metric in ['spearman', 'p_at_100', 'ndcg_at_100']:
    print(f"\n{metric}")
    print(results_table(metric, flat_methods).to_string())

---
# Step 5 — The cluster → merge → repeat hierarchy (with true values: the oracle)

**Up.** At level ℓ the graph has transition matrix `P_ℓ` and teleport vector `v_ℓ`. Cluster it into groups of
≤ `max_cluster` nodes. Given within-cluster shares `s` (summing to 1 per cluster) the super-node graph is

`P_{ℓ+1} = Sᵀ diag(s) P_ℓ S`,  `v_{ℓ+1} = Sᵀ v_ℓ`

— a weighted graph that keeps each super node's size (via `v`), its internal flow (self-loop) and its summed
in/out flow. If `s` are the *true* shares this is exact: PageRank of the coarse graph equals the summed
PageRank of each cluster (checked below). The partition itself is built once from edge counts, so it does not
depend on the model.

**Down.** Root mass 1; each node's mass = its share × its parent's mass, recursively.

**Oracle.** "Local-PR oracle": shares from PageRank *inside* each cluster, where the restart vector is the
node's teleport mass plus its inflow from outside the cluster (assuming outside sources carry their teleport
mass). It uses only local information, so it shows what the hierarchy can do without learning.

In [ ]:
# Cell 16: Clustering and hierarchy construction
def symmetric_weights(A):
    U = (A + A.T).tocsr(); U.setdiag(0); U.eliminate_zeros()
    return U

def rcm_chunks(U, idx, max_size):
    """Split idx into chunks of <= max_size following a bandwidth-reducing order (keeps neighbours together)."""
    sub = U[idx][:, idx].tocsr()
    perm = reverse_cuthill_mckee(sub, symmetric_mode=True)
    return [idx[perm[i:i + max_size]] for i in range(0, len(idx), max_size)]

def split_cluster(U, idx, max_size, res, depth, seed, method):
    if len(idx) <= max_size:
        return [idx]
    if method == 'rcm' or depth >= 6:
        return rcm_chunks(U, idx, max_size)
    sub = U[idx][:, idx].tocsr()
    if sub.nnz == 0:
        return rcm_chunks(U, idx, max_size)
    comms = nx.community.louvain_communities(nx.from_scipy_sparse_array(sub), weight='weight',
                                             resolution=res, seed=seed)
    if len(comms) <= 1:
        return rcm_chunks(U, idx, max_size)
    out = []
    for cm in comms:
        out.extend(split_cluster(U, idx[np.fromiter(cm, dtype=np.int64)], max_size, res * 2, depth + 1, seed, method))
    return out

def partition_level(A, max_size, seed=0, method='louvain'):
    n = A.shape[0]
    if n <= max_size:
        return np.zeros(n, dtype=np.int64)
    U = symmetric_weights(A)
    parts = split_cluster(U, np.arange(n), max_size, 1.0, 0, seed, method)
    min_pack = max(2, max_size // 10)                        # pack tiny clusters / isolated nodes together
    big = [p for p in parts if len(p) >= min_pack]
    small = [p for p in parts if len(p) < min_pack]
    if small:
        big += rcm_chunks(U, np.concatenate(small), max_size)
    if len(big) > n / 1.5:                                   # guarantee the graph shrinks
        big = rcm_chunks(U, np.arange(n), max_size)
    labels = np.empty(n, dtype=np.int64)
    for k, p in enumerate(big):
        labels[p] = k
    return labels

def build_hierarchy(A, max_size=CFG['max_cluster'], method=CFG['cluster_method'], seed=0, max_levels=20):
    """List of levels; level ℓ has n nodes, labels -> parent index, S (n x k membership). Last level has k = 1."""
    if method == 'auto':
        method = 'louvain' if A.shape[0] <= CFG['louvain_max_nodes'] else 'rcm'
    levels, Al = [], A.astype(np.float64).tocsr()
    for _ in range(max_levels):
        n = Al.shape[0]
        labels = partition_level(Al, max_size, seed, method)
        k = int(labels.max()) + 1
        S = sp.csr_matrix((np.ones(n), (np.arange(n), labels)), shape=(n, k))
        levels.append(dict(n=n, k=k, labels=labels, S=S))
        if k == 1:
            break
        Al = (S.T @ Al @ S).tocsr()
    else:
        raise RuntimeError("hierarchy did not reach a single root")
    return levels

def coarsen(P, s, lev):
    S = lev['S']
    return (S.T @ sp.diags(s) @ P @ S).tocsr()

def true_level_masses(pr, H):
    m = [pr]
    for lev in H:
        m.append(lev['S'].T @ m[-1])
    return m                                                # m[ℓ] has length H[ℓ]['n']; m[-1] == [1.0]

def true_log_shares(pr, H):
    m = true_level_masses(pr, H)
    return [np.log(m[l] / m[l + 1][H[l]['labels']]) for l in range(len(H))]

def run_hierarchy(rec, H, share_fn, collect=False, true_logs=None):
    """Bottom-up: shares at each level build the next level. Returns per-level log-shares (+ level batches)."""
    P, v = rec.P, rec.v
    logs, batches = [], []
    for l, lev in enumerate(H):
        s = np.maximum(np.asarray(share_fn(l, P, v, lev['labels']), dtype=float), 1e-300)
        logs.append(np.log(s))
        if collect:
            batches.append(make_batch(P, v, lev['labels'], true_logs[l]))
        if l < len(H) - 1:
            P = coarsen(P, s, lev); v = lev['S'].T @ v
    return (logs, batches) if collect else logs

def top_down(logs, H):
    """Returns predicted masses for every level (index 0 = original nodes)."""
    logm = np.zeros(1)
    masses = [np.ones(1)]
    for l in range(len(H) - 1, -1, -1):
        logm = logs[l] + logm[H[l]['labels']]
        masses.insert(0, np.exp(logm))
    return masses

def local_pr_share(P, v, labels, alpha=ALPHA, iters=60):
    n = len(labels); k = int(labels.max()) + 1
    coo = P.tocoo()
    intra = labels[coo.row] == labels[coo.col]               # self-loops count as intra flow
    Pi = sp.csr_matrix((coo.data[intra], (coo.row[intra], coo.col[intra])), shape=(n, n))
    PiT = Pi.T.tocsr()
    ext = np.bincount(coo.col[~intra], weights=coo.data[~intra] * v[coo.row[~intra]], minlength=n)
    pers = (1 - alpha) * v + alpha * ext + 1e-300
    r_hat = pers / np.bincount(labels, weights=pers, minlength=k)[labels]
    stay = alpha * np.asarray(Pi.sum(1)).ravel()
    x = r_hat.copy()
    for _ in range(iters):
        lost = x * (1 - stay)
        x = alpha * (PiT @ x) + r_hat * np.bincount(labels, weights=lost, minlength=k)[labels]
    return x / np.bincount(labels, weights=x, minlength=k)[labels]

def oracle_share_fn(l, P, v, labels):
    return local_pr_share(P, v, labels)

In [ ]:
# Cell 17: Step 5 — build hierarchies for the test suite, exact-aggregation check, oracle evaluation
HIER = {}
for rec in test_suite:                      # every test graph, so base vs new is compared on all of them
    t = time.time()
    HIER[rec.name] = build_hierarchy(rec.A)
    rec['hier_time'] = time.time() - t
    print(f"{rec.name:22s} levels: {' -> '.join(str(l['n']) for l in HIER[rec.name])} -> 1"
          f"   ({rec['hier_time']:.1f}s)")

# Sanity check: with the TRUE shares, coarse PageRank == summed fine PageRank (exact aggregation)
rec = next(r for r in test_suite if r.name in HIER)
H = HIER[rec.name]
tl = true_log_shares(rec.pr, H)
P1 = coarsen(rec.P, np.exp(tl[0]), H[0]); v1 = H[0]['S'].T @ rec.v
pr1, _, _ = pagerank_power(P1, v1, tol=1e-13)
print(f"\nExact-aggregation check on {rec.name}: L1(coarse PR, summed PR) = "
      f"{np.abs(pr1 - true_level_masses(rec.pr, H)[1]).sum():.2e}  (should be ~1e-12)")

for rec in test_suite:
    if rec.name not in HIER:
        continue
    pred = top_down(run_hierarchy(rec, HIER[rec.name], oracle_share_fn), HIER[rec.name])[0]
    log_result(rec, 'Hier: local-PR oracle', pred, refine=0)
print("\nSpearman — hierarchy oracle vs flat methods")
print(results_table('spearman', flat_methods + ['Hier: local-PR oracle']).to_string())

---
# Step 6 — The GNN inside the hierarchy: Stages A, B, C + final refinement

The **same** `PageRankPropGNN` (same weights) scores the nodes of every cluster at every level; the softmax inside a
cluster gives the split of the parent's mass. The representative of a cluster is simply its arg-max node.

* **Stage A — warm-up:** levels built from *true* shares, targets = true shares.
* **Stage B — own mistakes:** levels built bottom-up from the model's *own* (detached) shares, targets = true shares
  (global truth summed to that level). Repeated for a few rounds.
* **Stage C — end-to-end:** full up-and-down pass; loss on final `log(N·PR)` vs true global PageRank (+ ranking) plus
  per-level checks. Gradients flow through the mass-splitting products at every level into the one shared `W`;
  the coarse graphs themselves are built from detached shares (the winner/structure is a hard choice).

In [ ]:
# Cell 18: Hierarchy training graphs
rng_h = np.random.default_rng(11)
hier_train = []
for fam in CFG['families']:
    for _ in range(CFG['hier_train_per_family']):
        n = int(rng_h.integers(*CFG['hier_train_n']))
        r = Rec(make_record(f"htrain_{fam}", fam, make_graph(fam, n, rng_h)))
        r['H'] = build_hierarchy(r.A)
        r['true_logs'] = true_log_shares(r.pr, r['H'])
        hier_train.append(r)
print(f"{len(hier_train)} hierarchy training graphs; levels per graph: "
      f"{[len(r['H']) for r in hier_train]}")

def gnn_share_fn(model):
    model.eval()
    @torch.no_grad()
    def fn(l, P, v, labels):
        b = collate([make_batch(P, v, labels)])
        lp = group_logsoftmax(model(b['x'], b['ei'], b['ew']), b['g'], b['G'])
        return torch.exp(lp.double()).cpu().numpy()
    return fn

def true_share_fn(rec):
    return lambda l, P, v, labels: np.exp(rec['true_logs'][l])

In [ ]:
# Cell 19: Stage A (warm-up) and Stage B (own mistakes)
set_seed(3)
hier_model = PageRankPropGNN(hidden=CFG['hidden'], K=K_PROP, dropout=CFG['dropout']).to(device)
hier_model.load_state_dict(flat_model.state_dict())          # start from the flat model's weights
opt_h = torch.optim.AdamW(hier_model.parameters(), lr=1e-3, weight_decay=1e-5)
SNAPSHOTS, HIER_HIST = {}, {}

stageA_batches = []
for r in hier_train:
    _, bs = run_hierarchy(r, r['H'], true_share_fn(r), collect=True, true_logs=r['true_logs'])
    stageA_batches += bs
print(f"Stage A: {len(stageA_batches)} level batches")
HIER_HIST['A'] = train_on_batches(hier_model, stageA_batches, CFG['stageA_epochs'], opt_h, tag='[stage A]')
SNAPSHOTS['A'] = copy.deepcopy(hier_model.state_dict())

HIER_HIST['B'] = []
for rnd in range(CFG['stageB_rounds']):
    stageB_batches = []
    for r in hier_train:
        _, bs = run_hierarchy(r, r['H'], gnn_share_fn(hier_model), collect=True, true_logs=r['true_logs'])
        stageB_batches += bs
    HIER_HIST['B'] += train_on_batches(hier_model, stageB_batches, CFG['stageB_epochs'], opt_h, tag=f'[stage B r{rnd+1}]')
SNAPSHOTS['A+B'] = copy.deepcopy(hier_model.state_dict())

In [ ]:
# Cell 20: Stage C (end-to-end through all levels)
def hier_forward_train(model, rec):
    """One differentiable up-and-down pass. Returns (final log-PR per node, list of per-level losses)."""
    H = rec['H']; P, v = rec.P, rec.v
    lps, level_losses = [], []
    for l, lev in enumerate(H):
        b = collate([make_batch(P, v, lev['labels'], rec['true_logs'][l])])
        lp = group_logsoftmax(model(b['x'], b['ei'], b['ew']), b['g'], b['G'])
        lps.append(lp); level_losses.append(share_loss(lp, b))
        if l < len(H) - 1:
            s = torch.exp(lp.detach().double()).cpu().numpy()
            P = coarsen(P, s, lev); v = lev['S'].T @ v
    logm = torch.zeros(1, device=device)
    for l in range(len(H) - 1, -1, -1):                       # top-down: products of shares (sums of logs)
        logm = lps[l] + logm[torch.as_tensor(H[l]['labels'], device=device)]
    return logm, level_losses

HIER_HIST['C'] = []
opt_c = torch.optim.AdamW(hier_model.parameters(), lr=5e-4, weight_decay=1e-5)
for ep in range(1, CFG['stageC_epochs'] + 1):
    hier_model.train(); tot = 0.0
    for i in np.random.permutation(len(hier_train)):
        r = hier_train[i]
        logm, lvl = hier_forward_train(hier_model, r)
        tgt = torch.as_tensor(np.log(r.pr), dtype=torch.float, device=device)
        whole = dict(g=torch.zeros(r.n, dtype=torch.long, device=device), gsize=torch.full((r.n,), r.n, device=device),
                     gstart=torch.zeros(1, dtype=torch.long, device=device), gorder=torch.arange(r.n, device=device))
        loss = (F.smooth_l1_loss(logm, tgt) + CFG['rank_weight'] * ranking_loss(logm, tgt, whole, CFG['rank_pairs'])
                + CFG['level_loss_weight'] * torch.stack(lvl).mean())
        opt_c.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(hier_model.parameters(), 1.0); opt_c.step()
        tot += loss.item()
    HIER_HIST['C'].append(tot / len(hier_train))
    if ep == 1 or ep % max(1, CFG['stageC_epochs'] // 10) == 0 or ep == CFG['stageC_epochs']:
        print(f"[stage C] epoch {ep:03d}/{CFG['stageC_epochs']} | loss {HIER_HIST['C'][-1]:.4f}")
SNAPSHOTS['A+B+C'] = copy.deepcopy(hier_model.state_dict())

fig, axs = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, (k, h) in zip(axs, HIER_HIST.items()):
    ax.plot(range(1, len(h) + 1), h, color='#00b894', lw=2)
    ax.set_title(f"Stage {k} loss"); ax.set_xlabel("Epoch"); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 21: Evaluate the hierarchical GNN (+ final PageRank refinement) on the unseen suite
def refine(rec, x0, k):
    return x0 if k == 0 else pagerank_power(rec.P, rec.v, x0=x0, max_iter=k, tol=0.0)[0]

HIER_PRED = {}
for rec in test_suite:
    if rec.name not in HIER:
        continue
    H = HIER[rec.name]
    t = time.time()
    hier_model.load_state_dict(SNAPSHOTS['A+B+C'])
    logs = run_hierarchy(rec, H, gnn_share_fn(hier_model))
    est = top_down(logs, H)[0]
    rec['gnn_time'] = time.time() - t
    HIER_PRED[rec.name] = dict(logs=logs, est=est)
    oracle0 = top_down(run_hierarchy(rec, H, oracle_share_fn), H)[0]
    for k in CFG['refine_iters']:
        log_result(rec, 'Hier-GNN', refine(rec, est, k), refine=k, stage='A+B+C', csize=CFG['max_cluster'])
        if k:
            log_result(rec, 'Hier: local-PR oracle', refine(rec, oracle0, k), refine=k)
            log_result(rec, 'Power iter (uniform)', baseline_power(rec, k), refine=k)

no_refine = lambda df: df['refine'].fillna(0) == 0
main_methods = ['In-degree', 'Flat GraphSAGE (base)', 'Flat PRProp-GNN', 'Hier: local-PR oracle', 'Hier-GNN']
for metric in ['spearman', 'p_at_100', 'ndcg_at_100', 'top10pct']:
    print(f"\n{metric} (no refinement)")
    print(results_table(metric, main_methods, no_refine).to_string())

In [ ]:
# Cell 22: Headline chart — Spearman per family (no refinement)
df = pd.DataFrame(RESULTS)
df = df[no_refine(df) & df.method.isin(main_methods) & (df.family != 'er-orig')]
agg = df.groupby(['family', 'method']).spearman.mean().unstack()[main_methods]
fams = list(agg.index); w = 0.8 / len(main_methods)
fig, ax = plt.subplots(figsize=(12, 4.5))
for i, m in enumerate(main_methods):
    ax.bar(np.arange(len(fams)) + (i - (len(main_methods) - 1) / 2) * w, agg[m].values, width=w * 0.92,
           color=COLORS[m], label=m)
ax.set_xticks(range(len(fams))); ax.set_xticklabels(fams)
ax.set_ylabel("Spearman ρ vs true PageRank (mean over sizes)"); ax.set_ylim(min(0, agg.values.min() - 0.05), 1)
ax.set_title("Zero-shot ranking quality by graph family ('real' = SNAP graphs)")
ax.grid(True, axis='y', alpha=0.3); ax.legend(ncol=3, fontsize=9, frameon=False, loc='lower left')
plt.tight_layout(); plt.show()

---
# Step 7 — Per-level error, ablations, speed

In [ ]:
# Cell 23: Per-level error — where does error enter, and does it compound?
rows = []
for rec in test_suite:
    if rec.name not in HIER_PRED:
        continue
    H = HIER[rec.name]
    true_m = true_level_masses(rec.pr, H)
    tl = true_log_shares(rec.pr, H)
    pred_m = top_down(HIER_PRED[rec.name]['logs'], H)
    for l, lev in enumerate(H):
        # share quality at this level: mean within-cluster Spearman over clusters of >= 3 nodes
        ps, ts, lab = HIER_PRED[rec.name]['logs'][l], tl[l], lev['labels']
        rhos = [spearmanr(ps[lab == c], ts[lab == c])[0] for c in np.unique(lab) if (lab == c).sum() >= 3]
        rhos = [x for x in rhos if np.isfinite(x)]
        rows.append(dict(graph=rec.name, level=l, nodes=lev['n'],
                         share_spearman=np.mean(rhos) if rhos else np.nan,
                         share_l1=float(np.abs(np.exp(ps) - np.exp(ts)).sum() / lev['k']),
                         mass_l1_after_topdown=float(np.abs(pred_m[l] - true_m[l]).sum())))
per_level = pd.DataFrame(rows)
per_level.to_csv(os.path.join(CFG['results_dir'], 'per_level_error.csv'), index=False)
print(per_level.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
for name, g in per_level.groupby('graph'):
    ax.plot(g.level, g.mass_l1_after_topdown, marker='o', ms=8, lw=2, color='#00b894', alpha=0.35)
mean_curve = per_level.groupby('level').mass_l1_after_topdown.mean()
ax.plot(mean_curve.index, mean_curve.values, marker='o', ms=8, lw=2.5, color='#1757ac', label='Mean over test graphs')
ax.set_xlabel("Level (0 = original nodes)"); ax.set_ylabel("L1 error of level masses")
ax.set_title("Error after top-down splitting, per level"); ax.grid(True, alpha=0.3); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 24: Ablations — training stages × refinement, and cluster size
abl = []
for stage, sd in SNAPSHOTS.items():
    hier_model.load_state_dict(sd)
    for rec in test_suite:
        if rec.name not in HIER:
            continue
        est = top_down(run_hierarchy(rec, HIER[rec.name], gnn_share_fn(hier_model)), HIER[rec.name])[0]
        for k in CFG['refine_iters']:
            abl.append(dict(graph=rec.name, family=rec.family, stage=stage, refine=k, csize=CFG['max_cluster'],
                            **rank_metrics(refine(rec, est, k), rec.pr)))

hier_model.load_state_dict(SNAPSHOTS['A+B+C'])
for cs in CFG['ablation_cluster_sizes']:
    if cs == CFG['max_cluster']:
        continue
    for rec in test_suite:
        if rec.name not in HIER or rec.n > CFG['ablation_max_nodes']:
            continue
        Hc = build_hierarchy(rec.A, max_size=cs)
        est = top_down(run_hierarchy(rec, Hc, gnn_share_fn(hier_model)), Hc)[0]
        abl.append(dict(graph=rec.name, family=rec.family, stage='A+B+C', refine=0, csize=cs,
                        **rank_metrics(est, rec.pr)))
abl = pd.DataFrame(abl)
abl['kind'] = abl.family.map(lambda f: 'real' if f == 'real' else 'synthetic')
abl.to_csv(os.path.join(CFG['results_dir'], 'ablations.csv'), index=False)
print("Stages × refinement (mean over held-out graphs)")
print(abl[abl.csize == CFG['max_cluster']].groupby(['kind', 'stage', 'refine'])[['spearman', 'p_at_100', 'ndcg_at_100', 'l1']]
      .mean().round(4).to_string())
print("\nCluster size (stage A+B+C, no refinement)")
print(abl[(abl.stage == 'A+B+C') & (abl.refine == 0)].groupby(['kind', 'csize'])[['spearman', 'p_at_100', 'ndcg_at_100', 'l1']]
      .mean().round(4).to_string())

In [ ]:
# Cell 25: Speed — power-iteration steps to reach a target accuracy from each starting point
target = CFG['target_err']
speed, curves = [], {}
for rec in test_suite:
    if rec.name not in HIER_PRED:
        continue
    H = HIER[rec.name]
    inits = {'Uniform (standard PageRank)': rec.v,
             'In-degree': baseline_indegree(rec) / baseline_indegree(rec).sum(),
             'Flat PRProp-GNN': predict_flat(flat_model, rec),
             'Hier: local-PR oracle': top_down(run_hierarchy(rec, H, oracle_share_fn), H)[0],
             'Hier-GNN': HIER_PRED[rec.name]['est']}
    t = time.time(); pagerank_power(rec.P, rec.v, tol=target); t_pr = time.time() - t
    for name, x0 in inits.items():
        _, _, hist = pagerank_power(rec.P, rec.v, x0=x0, ref=rec.pr, tol=0.0, max_iter=200)
        hist = np.array(hist)
        hit = np.nonzero(hist < target)[0]
        speed.append(dict(graph=rec.name, init=name, start_l1=hist[0],
                          iters_to_target=int(hit[0]) if len(hit) else np.nan))
        curves.setdefault(rec.name, {})[name] = hist
    speed.append(dict(graph=rec.name, init='(wall clock) power iteration to target, s', start_l1=np.nan,
                      iters_to_target=round(t_pr, 3)))
    speed.append(dict(graph=rec.name, init='(wall clock) hierarchy build + GNN, s', start_l1=np.nan,
                      iters_to_target=round(rec['hier_time'] + rec['gnn_time'], 3)))
speed = pd.DataFrame(speed)
speed['kind'] = speed.graph.map({r.name: graph_kind(r) for r in test_suite})
speed.to_csv(os.path.join(CFG['results_dir'], 'speed.csv'), index=False)
print(speed.pivot_table(index='graph', columns='init', values='iters_to_target', aggfunc='first').to_string())

big = max((r for r in test_suite if r.name in curves), key=lambda r: r.n)
init_colors = {'Uniform (standard PageRank)': '#9c9c9c', 'In-degree': '#f2a600', 'Flat PRProp-GNN': '#257de0',
               'Hier: local-PR oracle': '#ee7fac', 'Hier-GNN': '#00b894'}
fig, ax = plt.subplots(figsize=(8, 4.5))
for name, h in curves[big.name].items():
    ax.semilogy(np.arange(len(h)), h, lw=2, color=init_colors[name], label=name)
ax.axhline(target, color='#666666', ls='--', lw=1)
ax.set_xlabel("Power-iteration steps"); ax.set_ylabel("L1 distance to true PageRank")
ax.set_title(f"Warm-start convergence on {big.name}"); ax.grid(True, alpha=0.3, which='both')
ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(CFG['results_dir'], 'warm_start_convergence.png'), dpi=150); plt.show()

---
# Final comparison — base code vs new additions, on real and synthetic graphs
*Base* = the original notebook's GraphSAGE (Cells 3–6). *New* = the flat PRProp-GNN (Steps 2–4) and the
hierarchical GNN (Steps 5–6), with and without the final 10-step refinement. In-degree and the local-PR oracle
are shown as reference points. Everything is also written to `results/`.

In [ ]:
# Cell 26: Base vs new — metric matrices, deltas, chart, CSVs
METRICS = ['spearman', 'kendall', 'top10pct', 'p_at_100', 'ndcg_at_100', 'l1']
COMPARE = {'Base: GraphSAGE (original)': ('Flat GraphSAGE (base)', 0),
           'New: Flat PRProp-GNN':       ('Flat PRProp-GNN', 0),
           'New: Hier-GNN':              ('Hier-GNN', 0),
           'New: Hier-GNN + 10 refine':  ('Hier-GNN', 10),
           'Ref: In-degree':             ('In-degree', 0),
           'Ref: Hier local-PR oracle':  ('Hier: local-PR oracle', 0)}
KIND_ORDER = ['real', 'synthetic: held-out families', 'synthetic: original ER']

all_df = pd.DataFrame(RESULTS)
all_df['refine'] = all_df['refine'].fillna(0).astype(int)
all_df.to_csv(os.path.join(CFG['results_dir'], 'all_metrics.csv'), index=False)
parts = []
for label, (method, k) in COMPARE.items():
    sub = all_df[(all_df.method == method) & (all_df.refine == k)].drop_duplicates('graph', keep='last').copy()
    sub['model'] = label
    parts.append(sub)
cmp = pd.concat(parts, ignore_index=True)
cmp.to_csv(os.path.join(CFG['results_dir'], 'base_vs_new_per_graph.csv'), index=False)
models = list(COMPARE)

# 1) Matrix: every metric, mean per kind of graph
summary = cmp.groupby(['model', 'kind'])[METRICS].mean().unstack('kind')
summary = summary.reindex(index=models, columns=pd.MultiIndex.from_product([METRICS, [k for k in KIND_ORDER if k in cmp.kind.unique()]]))
summary.round(4).to_csv(os.path.join(CFG['results_dir'], 'base_vs_new_summary.csv'))
for metric in METRICS:
    print(f"\n=== {metric} (mean per kind of graph; lower is better for l1) ===")
    print(summary[metric].round(4).to_string())

# 2) Matrix: Spearman and precision@100 per graph
for metric in ['spearman', 'p_at_100']:
    print(f"\n=== {metric} per graph ===")
    print(cmp.pivot_table(index=['kind', 'graph'], columns='model', values=metric)[models].round(4).to_string())

# 3) New minus base, per kind (positive = new is better; for l1, negative = better)
delta = []
for kind in [k for k in KIND_ORDER if k in cmp.kind.unique()]:
    base = cmp[(cmp.model == models[0]) & (cmp.kind == kind)].set_index('graph')[METRICS]
    for label in models[1:4]:
        new = cmp[(cmp.model == label) & (cmp.kind == kind)].set_index('graph')[METRICS]
        common = base.index.intersection(new.index)
        if len(common):
            delta.append(dict(kind=kind, comparison=f"{label} − base", graphs=len(common),
                              **(new.loc[common] - base.loc[common]).mean().round(4).to_dict()))
delta = pd.DataFrame(delta)
delta.to_csv(os.path.join(CFG['results_dir'], 'base_vs_new_delta.csv'), index=False)
print("\n=== New minus base (mean over the graphs both were run on) ===")
print(delta.to_string(index=False))

# 4) Which model ranks best on each kind of graph (Spearman, no refinement, learned + reference models)
print("\n=== Best model by Spearman (no refinement) ===")
no_ref = [m for m in models if 'refine' not in m]
for kind in [k for k in KIND_ORDER if k in cmp.kind.unique()]:
    sc = summary['spearman'][kind].reindex(no_ref).dropna()
    print(f"{kind:32s} best: {sc.idxmax()} ({sc.max():.3f}) | base: {sc.get(models[0], float('nan')):.3f}")

# 5) Chart: Spearman per kind of graph, base vs new
plot_models = models[:4] + ['Ref: In-degree']
pcolors = {'Base: GraphSAGE (original)': '#f47230', 'New: Flat PRProp-GNN': '#257de0', 'New: Hier-GNN': '#00b894',
           'New: Hier-GNN + 10 refine': '#008a17', 'Ref: In-degree': '#9c9c9c'}
kinds = [k for k in KIND_ORDER if k in cmp.kind.unique()]
w = 0.8 / len(plot_models)
fig, ax = plt.subplots(figsize=(12, 4.8))
for i, m in enumerate(plot_models):
    vals = [summary['spearman'][k].get(m, np.nan) for k in kinds]
    ax.bar(np.arange(len(kinds)) + (i - (len(plot_models) - 1) / 2) * w, vals, width=w * 0.92, color=pcolors[m], label=m)
ax.set_xticks(range(len(kinds))); ax.set_xticklabels(kinds)
ax.set_ylabel("Spearman ρ vs true PageRank (mean)"); ax.set_ylim(min(0, np.nanmin(summary['spearman'].values) - 0.05), 1.0)
ax.set_title("Base code vs new additions — real SNAP graphs vs synthetic graphs")
ax.grid(True, axis='y', alpha=0.3); ax.legend(ncol=3, fontsize=9, frameon=False, loc='lower left')
plt.tight_layout(); plt.savefig(os.path.join(CFG['results_dir'], 'base_vs_new_spearman.png'), dpi=150); plt.show()
print(f"\nSaved: {sorted(os.listdir(CFG['results_dir']))}")